In [ ]:
import os
import cv2
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def process_video(path):
    """
    Process a single video frame by frame to compute running sum, sum of squares, and pixel count.
    Returns: (sum_rgb, sumsq_rgb, pixel_count)
    """
    cap = cv2.VideoCapture(path)
    sum_rgb = np.zeros(3, dtype=np.float64)
    sumsq_rgb = np.zeros(3, dtype=np.float64)
    count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Convert to RGB and float
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32)
        pixels = frame.reshape(-1, 3)

        # Accumulate
        sum_rgb += pixels.sum(axis=0)
        sumsq_rgb += np.square(pixels).sum(axis=0)
        count += pixels.shape[0]
    cap.release()
    return sum_rgb, sumsq_rgb, count

def compute_mean_std_from_videos(directory, max_workers=None):
    """
    Compute per-channel mean and std from all .mp4 videos in the given directory.
    :param directory: Directory containing .mp4 videos.
    :param max_workers: Number of threads for parallel processing.
    :return: Dictionary with mean, std, and BGR conversion flag.
    """
    video_files = [os.path.join(directory, f)
                   for f in os.listdir(directory) if f.endswith('.mp4')]
    if not video_files:
        raise ValueError("No .mp4 files found in the specified directory.")

    # Global accumulators
    total_sum = np.zeros(3, dtype=np.float64)
    total_sumsq = np.zeros(3, dtype=np.float64)
    total_count = 0

    # Use thread pool to process multiple videos in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_video, path): path for path in video_files}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Processing videos"):
            sum_rgb, sumsq_rgb, count = fut.result()
            total_sum += sum_rgb
            total_sumsq += sumsq_rgb
            total_count += count

    # Online formula to calculate mean and std
    mean = total_sum / total_count
    var = (total_sumsq / total_count) - np.square(mean)
    std = np.sqrt(var)

    img_norm_cfg = {
        'mean': mean.tolist(),
        'std': std.tolist(),
        'to_bgr': False
    }
    return img_norm_cfg

directory = '../RGB/clips'
norm_cfg = compute_mean_std_from_videos(directory, max_workers=32)
print(norm_cfg)

In [ ]:
{'mean': [66.15693065312055, 59.39387486980162, 67.08884259986877], 'std': [62.46999995594521, 57.96686051956861, 58.18312924970111], 'to_bgr': False}